# Group Relative Policy Optimization (GRPO) with veRL on Amazon SageMaker Training jobs
## Lab 2 - GRPO training

**Stage: Training.** In this notebook we post-train `Qwen/Qwen3-4B` with GRPO on the dataset we prepared in Lab 1, using a SageMaker training job on one `ml.g6e.12xlarge` instance.

### What you will do

| Step | What happens | Input | Output |
| --- | --- | --- | --- |
| 1 | Install the notebook's Python dependencies | `requirements.txt` | Packages in the kernel |
| 2 | Resolve the SageMaker session, execution role, and default bucket | Your Studio environment | `role`, `bucket_name`, `region` |
| 3 | Find the training image in your account's ECR registry | The `verl-grpo` repository, tag `workshop` | `TRAINING_IMAGE_URI`, pinned by digest |
| 4 | Derive the data channels and output path | The session's bucket | `train_channel`, `validation_channel`, `output_path` |
| 5 | Set the GRPO hyperparameters | -- | `hyperparameters`, 25 entries |
| 6 | Set the container environment variables | -- | `environment` |
| 7 | Configure and submit the training job | Everything above | A job named `train-qwen3-4b-grpo-<timestamp>` |
| 8 | Monitor the job and read its metrics | The job name prefix | Status, then the streamed training log |

**Submit first, read second.** Steps 1 to 7 take about fifteen minutes to run and nothing in them requires you to have read the explanations. Once the job is submitted it waits for SageMaker to provision the instance -- about an hour in a Workshop Studio test run -- and then trains for about 100 minutes. Get it submitted, then come back to the text while it queues.

## Prerequisites

### Step 1 - Install requirements

**What this does:** installs the notebook's Python dependencies into the kernel.

**Input:** `./requirements.txt`. **Output:** pip's install log. A note that you may need to restart the kernel is normal; ignore it unless an import below fails.

In [ ]:
%pip install -r ./requirements.txt --upgrade

### Step 2 - Resolve the SageMaker session

**What this does:** creates a SageMaker session and reads the three things every later step uses: the execution role this notebook runs as, the account's default SageMaker bucket, and the region.

**Input:** nothing -- all three come from the Studio environment. **Output:** three printed lines. The role ARN should contain `SageMakerExecutionRole`, the bucket should be `sagemaker-<region>-<account id>`, and the region should match your event. If the bucket did not exist yet, the SDK creates it and logs that it did.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()

sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

sm_client = boto3.client("sagemaker")

sess = Session(default_bucket=sagemaker_session_bucket)

bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
region = sess.boto_region_name

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {region}")

***

## Group Relative Policy Optimization
GRPO replaces the reward model that PPO needs with something much cheaper. For each prompt it samples a *group* of completions, scores each one, and then uses the spread within that group as the baseline: completions that beat their group's average get reinforced, completions that fall short get discouraged. There is no separate critic network to train, which is what makes it practical at this size.

That design has a consequence for the infrastructure, and it is the reason this lab needs a different container from the other options in this solution. A GRPO step does three different jobs on the same GPUs:

| role | what it does | engine |
| --- | --- | --- |
| rollout | samples `rollout_n` completions per prompt | vLLM |
| actor | the policy being trained | FSDP |
| reference | frozen copy, for the KL penalty | FSDP |

Generation dominates the step, so the rollout engine has to be a real inference server rather than `model.generate` in a loop -- hence vLLM. The actor and reference are sharded with FSDP. veRL coordinates all of it with Ray, and hands the sampled completions to a reward function to score.

The reward function is the part we did not have to write. As we saw in Lab 1, the `data_source` field on each row routes it to veRL's built-in GSM8K scorer, which checks the model's `####` answer against the ground truth. Correct answer, reward 1. Wrong answer, reward 0. That is the entire signal the policy learns from.

### Step 3 - Find the training image

**What this does:** looks up the workshop's veRL training image in your account's Amazon ECR registry and pins it by digest.

**Input:** an ECR repository whose name contains `verl-grpo`, and the tag `workshop`. **Output:** one line, `training image: <account>.dkr.ecr.<region>.amazonaws.com/<stack>-verl-grpo@sha256:...`. If the cell raises instead, the image build has not produced an image yet; the Prerequisites page has the commands to check it, and nothing later in this lab can run without it.

veRL, vLLM, Ray, FSDP, and a matching CUDA build have to agree with each other, so this lab does not use a SageMaker Deep Learning Container. It uses an image built from veRL's own published base image with the SageMaker `/opt/ml` contract added on top -- the `container/` directory in this option holds the `Dockerfile`, the pinned base image digest, and a build-time check that imports veRL, Ray, vLLM, and torch and asserts their versions.

That image is built into your own account when the workshop provisions your environment, so there is nothing to build here and no URI to paste. The cell below finds it in Amazon ECR and pins it by digest, so every step of this run trains on exactly the image it inspected. SageMaker pulls training images from ECR, which is why the image has to live in the same account and region as the job.

If you are running these notebooks outside the workshop, build the image yourself from `container/Dockerfile`, push it to ECR in your own account, and set `GRPO_TRAINING_IMAGE_URI` to that URI -- an explicit value takes precedence over discovery. `container/base-image.lock.json` records the base image digest to build from and the versions it carries.

In [ ]:
import os

import boto3


def discover_training_image(name_contains="verl-grpo", tag="workshop"):
    """Resolve the workshop's training image from ECR in this account.

    The workshop builds this image into the account provisioned for you, in a
    repository named after the CloudFormation stack, so its URI cannot be written
    down in advance. Resolve it by digest rather than by tag: the tag is mutable,
    and pinning the digest means the whole run trains on exactly the image
    inspected here.

    Returns None rather than raising, so the caller can explain what to do.
    """
    ecr = boto3.client("ecr")
    matches = []
    for page in ecr.get_paginator("describe_repositories").paginate():
        for repo in page["repositories"]:
            if name_contains in repo["repositoryName"]:
                matches.append(repo)
    if len(matches) != 1:
        return None

    repo = matches[0]
    try:
        images = ecr.describe_images(
            repositoryName=repo["repositoryName"],
            imageIds=[{"imageTag": tag}],
        )["imageDetails"]
    except ecr.exceptions.ImageNotFoundException:
        return None
    if not images:
        return None
    return f"{repo['repositoryUri']}@{images[0]['imageDigest']}"


# An explicit value wins. That is how you point this lab at your own build.
TRAINING_IMAGE_URI = os.environ.get("GRPO_TRAINING_IMAGE_URI", "") or discover_training_image()

if not TRAINING_IMAGE_URI:
    raise ValueError(
        "Could not find the training image.\n\n"
        "In the workshop it is built into your account while the environment is "
        "provisioned, into an ECR repository whose name contains 'verl-grpo', tagged "
        "'workshop'. If it is not there the build may still be running, or may have "
        "failed; the Prerequisites page has the commands to check both.\n\n"
        "Outside the workshop, build container/Dockerfile, push it to ECR, and set "
        "GRPO_TRAINING_IMAGE_URI to the result. container/base-image.lock.json "
        "records the base image to build from."
    )

print(f"training image: {TRAINING_IMAGE_URI}")

### Step 4 - Data channels and output path

**What this does:** builds the S3 prefixes for the two input channels and the output location, from the same session values Lab 1 used.

**Input:** `bucket_name` and `default_prefix` from Step 2. **Output:** three lines -- the train and validation prefixes, ending in `/train/` and `/validation/`, and the output path `s3://<bucket>/grpo-verl`, under which the job will write its exported model and checkpoints.

These are the same S3 paths Lab 1 wrote to, re-derived rather than passed along. Note they are *prefixes*, not files: SageMaker mounts a channel as a directory, and the container resolves the parquet file inside it before handing the path to veRL.

In [ ]:
if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-grpo"
    output_path = f"s3://{bucket_name}/{default_prefix}/grpo-verl"
else:
    input_path = "datasets/llm-fine-tuning-grpo"
    output_path = f"s3://{bucket_name}/grpo-verl"

train_channel = f"s3://{bucket_name}/{input_path}/train/"
validation_channel = f"s3://{bucket_name}/{input_path}/validation/"

print(f"train:      {train_channel}")
print(f"validation: {validation_channel}")
print(f"output:     {output_path}")

## Step 5 - Hyperparameters

**What this does:** defines the GRPO configuration as a flat mapping of strings, which SageMaker will write into the container at `/opt/ml/input/config/hyperparameters.json`.

**Input:** none. **Output:** `25`, the number of entries. Every value is a string, because that is how SageMaker renders hyperparameters; `scripts/run_grpo.py` parses and range-checks each one inside the container before any GPU work starts, so a bad value fails in seconds rather than twenty minutes into a paid job.

The other options in this solution write an `args.yaml`, upload it, and mount it as a channel. This one passes hyperparameters the way SageMaker passes them natively, and `run_grpo.py` turns each entry into the corresponding veRL Hydra override, so the job's own console shows exactly what the trainer was asked to do.

The ones that matter most here:

- **`rollout_n: 8`** is the group size, the *G* in GRPO. Eight completions per prompt is the smallest group that gives a usable within-group baseline. It also means a step generates eight times as much text as it trains on, which is why generation dominates the wall clock.
- **`train_batch_size: 128`** against the 1280 rows from Lab 1 is exactly 10 steps.
- **`use_kl_loss` and `kl_loss_coef: 0.001`** keep the policy near the reference model. Without the penalty a GRPO run will happily learn to produce degenerate text that games the scorer.
- **`tensor_model_parallel_size: 2`** shards the vLLM rollout engine across 2 of the 4 GPUs.
- **`gpu_memory_utilization: 0.5`** caps what vLLM reserves for its KV cache, leaving room for the actor and reference shards on the same cards. This is the value to lower first if a run fails on GPU memory.
- **`optimizer_offload: true`** keeps optimizer state on the host, which is what makes a 4B actor fit alongside a rollout engine.
- **`test_freq: 10`** validates on step 10, the last one. veRL's default is `-1`, which means *never*, and a run left at the default reports a step-0 baseline with nothing to compare it against.

In [ ]:
hyperparameters = {
    # Algorithm
    "adv_estimator": "grpo",
    "rollout_n": "8",
    "use_kl_loss": "True",
    "kl_loss_coef": "0.001",
    "loss_agg_mode": "token-mean",
    # Model and sequence budget
    "actor_path": "Qwen/Qwen3-4B",
    "max_prompt_length": "1024",
    "max_response_length": "1024",
    # Batching
    "train_batch_size": "128",
    "ppo_mini_batch_size": "64",
    "use_dynamic_bsz": "True",
    "ppo_max_token_len_per_gpu": "8192",
    # Rollout engine
    "rollout_name": "vllm",
    "tensor_model_parallel_size": "2",
    "gpu_memory_utilization": "0.5",
    # Memory
    "param_offload": "False",
    "optimizer_offload": "True",
    # Schedule
    "total_epochs": "1",
    "save_freq": "10",
    "test_freq": "10",
    "max_actor_ckpt_to_keep": "2",
    "seed": "42",
    "logger": "[console]",
    # Topology. instance_count becomes trainer.nnodes; the GPUs per node come from
    # SM_NUM_GPUS, which SageMaker sets, so it is never hardcoded here.
    "instance_count": "1",
}

len(hyperparameters)

### Step 6 - Environment variables

**What this does:** sets three environment variables for the training container: two NCCL settings this instance family needs, and one that switches off a feature of the Ray launcher we do not use.

**Input:** none. **Output:** none; the cell defines `environment` and prints nothing.

The two NCCL variables are not optional on `ml.g6e.12xlarge`. The four L40S cards in that instance have no NVLink between them, and NCCL's peer-to-peer and shared-memory transports fail to initialise rather than falling back, so a job without these hangs during Ray cluster setup instead of reporting anything useful. Disabling them routes every intra-node collective through host-staged copies.

That costs real throughput -- it is part of why a step here takes around ten minutes -- and it is the price of this instance family. An instance with NVLink would not need them.

In [ ]:
environment = {
    "NCCL_P2P_DISABLE": "1",
    "NCCL_SHM_DISABLE": "1",
    # launcher.py starts a local Prometheus on the head node by default, which
    # means downloading a binary at job start and waiting up to five minutes if
    # that fails. veRL logs its metrics to the console, so it is not needed here.
    "launch_prometheus": "false",
}

## Step 7 - Configure and submit the training job

**What this does:** assembles a `ModelTrainer` from the image, code, compute, hyperparameters, environment, output, and checkpoint settings, then submits it with the two data channels.

**Input:** everything defined above. **Output:** the configuration cell prints nothing. The submit cell logs `Training Job Name: train-qwen3-4b-grpo-<timestamp>` and returns straight away, because `wait=False` -- the job runs on its own instance, not in this notebook.

Three details in the configuration below are load-bearing.

`SourceCode` uploads `./scripts` and SageMaker mounts it at `/opt/ml/code`, so the container supplies the environment and the notebook supplies the code. That means you can edit `scripts/run_grpo.py` and resubmit without rebuilding the image.

`compression_type="NONE"` on the output config matters for Lab 3. It leaves the exported model as a browsable S3 prefix instead of a `model.tar.gz`, and the vLLM serving container reads the model from a mounted prefix. Compressing it here would mean unpacking 8 GB on every endpoint start.

There is no `distributed=` argument, and `SourceCode` takes a `command` rather than an `entry_script`. The other options pass `Torchrun()`; this one does not, because veRL distributes work with Ray. The command runs `scripts/launcher.py`, the standard Ray launcher for SageMaker training jobs from [aws-samples/sample-ray-on-amazon-sagemaker-training-jobs](https://github.com/aws-samples/sample-ray-on-amazon-sagemaker-training-jobs), copied into `scripts/` unmodified. It starts the Ray head, joins any worker instances, and then runs `scripts/entrypoint.py` on the head node with a connected Ray driver. Adding a torchrun launcher on top would start the entry point once per GPU and fight it.

One thing to know if you hit a `RoleValidationError` here: the SDK checks the execution role's permissions on the client side before it submits anything, and it is stricter than the API. A scoped-down role that can genuinely run training jobs can still be refused -- `cloudwatch:PutMetricData` is a common one to be missing. The error names the permissions it wants. The default SageMaker Studio execution role has them.

In [ ]:
from datetime import datetime, timezone

from sagemaker.train.configs import (
    CheckpointConfig,
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.model_trainer import ModelTrainer

job_prefix = "train-qwen3-4b-grpo"
# Identifies this run's checkpoint prefix. See checkpoint_config below.
run_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

source_code = SourceCode(
    source_dir="./scripts",
    # launcher.py forms the Ray cluster, then runs entrypoint.py on the head node.
    command="python launcher.py --entrypoint entrypoint.py",
)

compute_configs = Compute(
    instance_type="ml.g6e.12xlarge",
    instance_count=1,
    volume_size_in_gb=500,
    keep_alive_period_in_seconds=0,
)

model_trainer = ModelTrainer(
    role=role,
    training_image=TRAINING_IMAGE_URI,
    source_code=source_code,
    base_job_name=job_prefix,
    compute=compute_configs,
    # Three hours. Ten steps at roughly ten minutes each, plus model download and two
    # validation passes, fits comfortably. This is the only hard bound on what a hung
    # job can cost.
    stopping_condition=StoppingCondition(max_runtime_in_seconds=10800),
    hyperparameters=hyperparameters,
    environment=environment,
    output_data_config=OutputDataConfig(
        s3_output_path=output_path,
        compression_type="NONE",
    ),
    checkpoint_config=CheckpointConfig(
        # Scoped to this run. A prefix shared between runs is the problem here:
        # SageMaker syncs it into /opt/ml/checkpoints before training starts, veRL's
        # resume_mode defaults to "auto", and it resumes from whatever final step it
        # finds. A second run then trains nothing, skips the val_before_train
        # baseline, and leaves Lab 4 comparing one number against itself.
        s3_uri=f"{output_path}/checkpoints/{run_id}",
        local_path="/opt/ml/checkpoints",
    ),
)

Define the data channels and submit.

**Expected output:** a few INFO lines from the SDK ending with `Training Job Name: train-qwen3-4b-grpo-<timestamp>`. The cell returns in seconds. If it raises `RoleValidationError`, see the note above.

In [ ]:
from sagemaker.train.configs import InputData

data = [
    InputData(channel_name="train", data_source=train_channel),
    InputData(channel_name="validation", data_source=validation_channel),
]

model_trainer.train(input_data_config=data, wait=False)

### Step 8 - Monitor the job

**What this does:** looks the job up by name prefix, prints its status, then follows its log to completion.

**Input:** `job_prefix`. **Output:** two lines, `job: train-qwen3-4b-grpo-<timestamp>` and `status: InProgress (Pending)` -- or `(Starting)` if you ran it straight after submitting. Re-run the cell whenever you want a fresh status; it does not hold on to the submission handle, so it also works in a fresh kernel.

`wait=False` returned as soon as the job was submitted, so nothing in this notebook has been waiting on it. What happens next, and how long it takes:

| Secondary status | What SageMaker is doing | How long |
| --- | --- | --- |
| `Pending` | Provisioning an `ml.g6e.12xlarge`. The status message reads *Training job waiting for capacity*. | About an hour in a Workshop Studio test run; from minutes to several hours in other accounts. **Not billed.** |
| `Downloading` | Pulling the 14 GB image and the model weights onto the instance | A few minutes |
| `Training` | Baseline validation over the 256 rows, ten GRPO steps of about ten minutes each, final validation, then export | About 100 minutes |
| `Uploading` | Copying the exported model to `output_path` | A minute or two |

While it is `Pending`, this is the moment to go back and read Steps 5 to 7, and the workshop's Introduction page.

In [ ]:
from sagemaker.core.resources import TrainingJob

# Look the job up by name prefix rather than holding on to the submission handle,
# so this cell still works in a fresh kernel.
job = next(
    TrainingJob.get_all(
        name_contains=job_prefix,
        sort_by="CreationTime",
        sort_order="Descending",
    )
)

print(f"job:    {job.training_job_name}")
print(f"status: {job.training_job_status} ({job.secondary_status})")

The console output carries veRL's own per-step metrics. Three are worth watching as they scroll past:

- **`val-core/openai/gsm8k/acc/mean@1`** is the fraction of validation questions answered correctly. It appears twice -- once before step 1 (your baseline, typically 39% to 45%) and once after step 10 (typically near 76%). This is the number Lab 4 reads back.
- **`critic/score/mean`** on each `step:N` line is the average reward over that step's 1,024 sampled completions. It should trend upward. The `critic/` prefix is a naming leftover from PPO; there is no critic in this run.
- **`actor/kl_loss`** is the distance from the reference model, and should stay small.

**Expected output:** the job's log, streamed until it reaches `Completed`. Interrupting the cell stops the following, not the job; re-run it to pick the log up again.

In [ ]:
# Follow the job to completion. Interrupting this cell stops the following, not the job.
job.wait(logs=True)

***

### What to expect
Ten GRPO steps over 1280 prompts is 10,240 sampled completions, and it is enough to move GSM8K accuracy substantially -- runs of this exact configuration have gone from roughly 40% at step 0 to around 76% at step 10, a gain of more than thirty points. Two validated runs measured 39.45% to 76.56% and 44.14% to 76.17%. The step-0 figure is itself measured by sampling, so expect it to vary by a few points between runs; the step-10 figure is consistently near 76%.

It is worth being clear about what that number is and is not. It is a real measurement on held-out data, and it shows the pipeline works end to end. It is not a converged model: ten steps over a sixth of the training set was chosen so the lab finishes, not so the model stops improving. The full dataset is 7473 rows, which at this batch size is 58 steps.